# Analyse faciale : détection, embeddings et vérification

Ce notebook analyse le pipeline utilisé par l'application (`backend/`) :

| Étape | Modèle | Rôle |
|---|---|---|
| 1. Détection | **YuNet** (CNN léger, ONNX) | trouver les visages + 5 points de repère |
| 2. Alignement | `alignCrop` | redresser le visage (yeux horizontaux), 112×112 |
| 3. Embedding | **SFace** (réseau MobileFaceNet) | visage → vecteur de 128 nombres |
| 4. Comparaison | similarité cosinus | ≥ seuil → même personne |

**Données :** `data/dataset/person_XXX/img_XX.jpg`, téléchargées en ligne depuis **LFW**
(*Labeled Faces in the Wild*), découpées en `data/reference/` (galerie) et `data/test/`.

Installation : `pip install -r ../backend/requirements.txt matplotlib jupyter`

In [ ]:
import json, subprocess, sys
from itertools import combinations
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "backend"))   # on réutilise le code du backend

from detection import detect, ensure_models
from embeddings import aligned_face, embed_face

ensure_models()  # télécharge YuNet + SFace la première fois
print("OpenCV", cv2.__version__, "| données :", DATA)

## 1. Télécharger les données en ligne

Le script récupère LFW (~200 Mo la première fois, puis en cache), choisit des personnes au hasard et crée
`person_001/img_01.jpg`, …  Relancez avec d'autres paramètres pour plus de personnes.

In [ ]:
if not (DATA / "dataset").exists():
    subprocess.run([sys.executable, str(DATA / "download_dataset.py"),
                    "--people", "20", "--max", "8", "--ref", "3"], check=True)

NAMES = json.loads((DATA / "persons.json").read_text(encoding="utf-8"))
EXT = {".jpg", ".jpeg", ".png"}

def load_split(folder):
    items = []
    for d in sorted(p for p in folder.iterdir() if p.is_dir()):
        for f in sorted(d.iterdir()):
            if f.suffix.lower() in EXT:
                items.append((d.name, f))
    return items

dataset = load_split(DATA / "dataset")
persons = sorted({p for p, _ in dataset})
print(f"{len(persons)} personnes, {len(dataset)} images")
for pid in persons[:10]:
    print(f"  {pid}  {NAMES.get(pid, pid):28s} {sum(p == pid for p, _ in dataset)} images")

In [ ]:
# Aperçu : 3 images des 6 premières personnes
fig, axes = plt.subplots(3, 6, figsize=(13, 7))
for col, pid in enumerate(persons[:6]):
    files = [f for p, f in dataset if p == pid][:3]
    for row in range(3):
        ax = axes[row, col]; ax.axis("off")
        if row < len(files):
            ax.imshow(cv2.cvtColor(cv2.imread(str(files[row])), cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(f"{pid}\n{NAMES.get(pid, '')}", fontsize=9)
plt.tight_layout(); plt.show()

## 2. Détection avec YuNet

Chaque détection = boîte `(x, y, w, h)`, 5 points de repère (yeux, nez, coins de la bouche) et un score.

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, (pid, f) in zip(axes, dataset[::max(1, len(dataset) // 6)]):
    img = cv2.imread(str(f)); vis = img.copy()
    for face in detect(img):
        x, y, w, h = face[:4].astype(int)
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)
        for k in range(5):
            cv2.circle(vis, (int(face[4 + 2*k]), int(face[5 + 2*k])), 3, (0, 0, 255), -1)
        ax.set_title(f"score {face[14]:.2f}", fontsize=9)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.show()

## 3. Alignement et calcul des embeddings

In [ ]:
embs, labels, files, aligned, misses = [], [], [], [], []
for pid, f in dataset:
    img = cv2.imread(str(f))
    faces = detect(img)
    if len(faces) == 0:
        misses.append(f); continue
    embs.append(embed_face(img, faces[0])); labels.append(pid); files.append(f)
    if len(aligned) < 8:
        aligned.append(aligned_face(img, faces[0]))

E = np.array(embs); y = np.array(labels)
print(f"Embeddings : {E.shape}  |  visages non détectés : {len(misses)} / {len(dataset)}")

fig, axes = plt.subplots(1, len(aligned), figsize=(14, 2.2))
for ax, a in zip(axes, aligned):
    ax.imshow(cv2.cvtColor(a, cv2.COLOR_BGR2RGB)); ax.axis("off")
fig.suptitle("Visages alignés 112×112 (entrée de SFace)"); plt.show()

## 4. Visualiser l'espace des embeddings (ACP 2D)

Les 128 dimensions sont projetées en 2D. Les photos d'une même personne doivent former un groupe.

In [ ]:
from sklearn.decomposition import PCA

P = PCA(n_components=2).fit_transform(E)
plt.figure(figsize=(8, 6))
for pid in persons[:10]:
    m = y == pid
    plt.scatter(P[m, 0], P[m, 1], s=40, label=NAMES.get(pid, pid))
plt.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.title("Embeddings SFace projetés en 2D (10 premières personnes)")
plt.tight_layout(); plt.show()

## 5. Matrice de similarité cosinus

In [ ]:
S = E @ E.T   # embeddings normalisés → produit scalaire = cosinus
plt.figure(figsize=(7, 6))
plt.imshow(S, cmap="viridis", vmin=-0.2, vmax=1); plt.colorbar(label="similarité")
ticks = [np.where(y == p)[0][0] for p in persons if (y == p).any()]
plt.xticks(ticks, [p[-3:] for p in persons if (y == p).any()], fontsize=7, rotation=90)
plt.yticks(ticks, [p[-3:] for p in persons if (y == p).any()], fontsize=7)
plt.title("Les blocs clairs sur la diagonale = même personne"); plt.show()

## 6. Paires « genuine » vs « imposteur », courbe ROC et choix du seuil

- **Genuine** : deux photos de la même personne
- **Imposteur** : deux photos de personnes différentes
- **FAR** (*False Acceptance Rate*) : imposteurs acceptés · **FRR** (*False Rejection Rate*) : vraies personnes refusées
- **EER** : point où FAR = FRR

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

i, j = np.triu_indices(len(E), k=1)
scores = S[i, j]
same = (y[i] == y[j]).astype(int)
gen, imp = scores[same == 1], scores[same == 0]
print(f"{len(gen)} paires genuine, {len(imp)} paires imposteur")

fpr, tpr, thr = roc_curve(same, scores)
auc = roc_auc_score(same, scores)
eer_i = np.argmin(np.abs(fpr - (1 - tpr)))
EER, EER_T = (fpr[eer_i] + 1 - tpr[eer_i]) / 2, thr[eer_i]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
bins = np.linspace(-0.3, 1, 40)
ax[0].hist(imp, bins, alpha=.6, density=True, label="imposteurs", color="orange")
ax[0].hist(gen, bins, alpha=.6, density=True, label="même personne", color="green")
ax[0].axvline(0.363, ls="--", c="k", label="seuil SFace 0.363")
ax[0].axvline(EER_T, ls=":", c="r", label=f"seuil EER {EER_T:.3f}")
ax[0].set_xlabel("similarité cosinus"); ax[0].legend()
ax[1].plot(fpr, tpr); ax[1].plot([0, 1], [0, 1], "k:", lw=.8)
ax[1].set_xlabel("FAR"); ax[1].set_ylabel("1 - FRR"); ax[1].set_title(f"ROC  (AUC = {auc:.3f})")
plt.show()
print(f"EER = {EER:.2%} au seuil {EER_T:.3f}")

In [ ]:
print(f"{'seuil':>6} | {'FAR':>7} | {'FRR':>7} | précision")
for t in [0.25, 0.30, 0.363, 0.40, 0.45, 0.50, round(float(EER_T), 3)]:
    far, frr = np.mean(imp >= t), np.mean(gen < t)
    acc = 1 - (far * len(imp) + frr * len(gen)) / (len(imp) + len(gen))
    print(f"{t:6.3f} | {far:7.2%} | {frr:7.2%} | {acc:.2%}")

## 7. Protocole de l'application : galerie `reference/` → images `test/`

Exactement ce que calcule `GET /api/evaluate` : chaque image de test est comparée au **meilleur score**
parmi les images de référence de chaque personne.

In [ ]:
import verification as V

gallery = V.Gallery()
res = V.evaluate(gallery)
for k in ["n_test_images", "verification_accuracy", "identification_rank1", "FAR", "FRR", "EER", "EER_threshold"]:
    print(f"{k:24s} {res[k]}")

## 8. Exemples de vérification

In [ ]:
rng = np.random.default_rng(0)
def show_pair(a, b, ax_pair):
    s = float(E[a] @ E[b]); ok = s >= 0.363
    for ax, k in zip(ax_pair, (a, b)):
        ax.imshow(cv2.cvtColor(cv2.imread(str(files[k])), cv2.COLOR_BGR2RGB)); ax.axis("off")
        ax.set_title(NAMES.get(y[k], y[k]), fontsize=8)
    ax_pair[0].text(1.05, -0.08, f"{s:.2f} → {'MÊME' if ok else 'DIFFÉRENT'}",
                    transform=ax_pair[0].transAxes, ha="center", color="green" if ok else "red", fontsize=10)

g_idx = np.where(same == 1)[0]; i_idx = np.where(same == 0)[0]
picks = list(rng.choice(g_idx, 3, replace=False)) + list(rng.choice(i_idx, 3, replace=False))
fig, axes = plt.subplots(2, 6, figsize=(14, 5.5))
for n, p in enumerate(picks):
    r, c = divmod(n, 3)
    show_pair(i[p], j[p], axes[r, 2*c:2*c+2])
fig.suptitle("Ligne 1 : même personne · Ligne 2 : personnes différentes"); plt.tight_layout(); plt.show()

In [ ]:
# Les erreurs les plus fortes (utile pour comprendre les limites)
worst_gen = g_idx[np.argsort(scores[g_idx])[:3]]     # même personne, score faible
worst_imp = i_idx[np.argsort(-scores[i_idx])[:3]]    # personnes différentes, score élevé
fig, axes = plt.subplots(2, 6, figsize=(14, 5.5))
for n, p in enumerate(list(worst_gen) + list(worst_imp)):
    r, c = divmod(n, 3)
    show_pair(i[p], j[p], axes[r, 2*c:2*c+2])
fig.suptitle("Cas difficiles : ligne 1 = même personne mal notée · ligne 2 = imposteurs proches")
plt.tight_layout(); plt.show()

## 9. Conclusions

- SFace sépare bien les identités : les scores « même personne » sont nettement au-dessus des imposteurs.
- Le seuil **0.363** (recommandé par OpenCV) est un bon point de départ ; ajustez-le selon le besoin :
  **sécurité** (contrôle d'accès) → seuil plus haut, **confort** (tri de photos) → seuil plus bas.
- Erreurs typiques : profil marqué, lunettes de soleil, faible résolution, forte différence d'âge.

**Pour aller plus loin :** ajouter vos propres personnes dans `data/dataset/person_XXX/`,
puis `python data/download_dataset.py --split-only` ; tester un détecteur de vivacité (anti-photo).

**Données personnelles :** les visages sont des données biométriques. Pour des personnes réelles
(étudiants, salariés), recueillir leur consentement et respecter la loi 09-08 (CNDP).